# 📓 Semana 8 · Dia 2 — Particionamento vs Liquid Clustering vs Z-ORDER

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (performance) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Estratégia de clustering aplicada + documentada |

---


## 📖 Teoria — As 3 estratégias

| Estratégia | Como | Quando |
|---|---|---|
| **Particionamento** | pastas fixas por coluna | colunas de baixa cardinalidade; filtro exato (ano) |
| **Liquid Clustering** (`CLUSTER BY`) | reordenamento adaptativo automático | padrão 2026; alta cardinalidade; múltiplos filtros |
| **Z-ORDER** (legado) | índice de ordenação multi-coluna | ⚠️ deprecado — use CLUSTER BY |

**Regra prática** (documentação Databricks):
- < 1 TB: geralmente não precisa de particionamento/clustering — só OPTIMIZE.
- > 1 TB com filtros frequentes: Liquid Clustering por 1–4 colunas.
- Particionar SÓ por data (baixa cardinalidade e pruning efetivo).


### 💻 Na prática — Liquid Clustering na prática

Crie uma tabela clusterizada e otimize.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.prata.fato_vendas_cluster (
  InvoiceNo STRING, StockCode STRING, sk_cliente LONG, sk_produto LONG,
  data_venda DATE, Country STRING, Quantity INT, UnitPrice DOUBLE, receita DOUBLE)
USING DELTA
CLUSTER BY (Country, data_venda);

In [ ]:
# Popular a partir do Bronze
from pyspark.sql.functions import to_date
df = (spark.table("workspace.bronze.vendas_bronze")
    .select("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "Country",
            to_date("InvoiceDate", "M/d/yyyy H:mm").alias("data_venda"))
    .withColumn("receita", col("Quantity") * col("UnitPrice")))
df.write.mode("overwrite").saveAsTable("workspace.prata.fato_vendas_cluster")
print("Populado:", spark.table("workspace.prata.fato_vendas_cluster").count())

In [ ]:
# OPTIMIZE + histograma de clustering
spark.sql("OPTIMIZE workspace.prata.fato_vendas_cluster")
spark.sql("ANALYZE TABLE workspace.prata.fato_vendas_cluster COMPUTE STATISTICS")
display(spark.sql("DESCRIBE DETAIL workspace.prata.fato_vendas_cluster"))

### 💻 Na prática — Benchmark de leitura

Compare leitura com e sem filtro na coluna de clustering.


In [ ]:
# Filtro na coluna cluster (Country) — pruning eficiente
t0 = time.time()
n1 = spark.sql("SELECT COUNT(*) FROM workspace.prata.fato_vendas_cluster WHERE Country = 'United Kingdom'").collect()[0][0]
t1 = time.time()
print(f"UK: {n1} linhas em {t1-t0:.2f}s (com clustering, lê menos arquivos)")

> 🎯 **Dica de prova**: Liquid Clustering (CLUSTER BY) é o padrão 2026; Z-ORDER está deprecado. Pergunta típica: coluna de alta cardinalidade → clustering, não particionamento.


## 🎯 Exercícios de fixação

**1.** Quando particionar vs clusterizar?

**2.** Rode OPTIMIZE na sua fato_vendas_cluster e compare o tempo de uma query antes/depois.

**3.** O que o histograma de clustering mostra?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Particionar vs cluster

Particionar: baixa cardinalidade + filtro exato (ex.: ano). Cluster: alta cardinalidade, múltiplas colunas de filtro, escrita frequente.

**2.** Benchmark

Anote o tempo antes (muitos arquivos) e depois (arquivos compactados e ordenados) — a melhora vem do pruning + menos arquivos.

**3.** Histograma

Mostra o quanto os dados estão clusterizados por coluna (0-1): próximo de 1 = bem organizado para pruning.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*